# DVC: Version Control for Data and Models

## The Core Question

You trained a turbulence model. The dataset is 2 GB. Your trained model is 500 MB. Your code is 200 KB. How do you version-control all three in one repository without bloating Git to 5 GB?

**DVC (Data Version Control)** extends Git to track large files. The file content lives in a remote storage (S3, GDrive, SSH); Git stores a small pointer. You get full version history for your data and models, just like code.

## Why DVC?

| Problem | Without DVC | With DVC |
|---------|-------------|----------|
| 2 GB dataset in Git | Repo bloats forever (Git never shrinks) | Git stores 100-byte pointer |
| Share data with colleague | Email attachment / Dropbox | `git clone` + `dvc pull` |
| Reproduce a result | "Which data file did you use?" | Git commit + DVC hash = exact data |
| Experiment tracking | "I changed the data and..." | Each DVC commit is a data version |

## How It Works

```
Your data file (2 GB)
        |
        v
dvc add data.csv
        |
        v
Git stores: data.csv.dvc (pointer with hash)
DVC remote:  actual 2 GB file (keyed by hash)
        |
        v
git commit + push
        |
        v
Colleague: git clone + dvc pull -> exact same data


In [ ]:
!pip install dvc -q 2>/dev/null
import subprocess
import os
import shutil

# Clean up from previous runs
dvc_dir = './dvc_demo'
if os.path.exists(dvc_dir):
    shutil.rmtree(dvc_demo, ignore_errors=True)
os.makedirs(dvc_dir)
os.chdir(dvc_dir)


## 1. Initialize DVC in a Project

**Concept**: DVC works alongside Git. Initialize both in the same directory. DVC adds a `.dvc/` config folder and tracks large files.


In [ ]:
# Initialize Git (if not already)
result = subprocess.run(['git', 'init'], capture_output=True, text=True)
print('Git init:', result.stdout.strip() or 'already initialized')

# Initialize DVC
result = subprocess.run(['dvc', 'init', '--no-scm'], capture_output=True, text=True)
print('DVC init:', result.stdout.strip() if result.returncode == 0 else result.stderr.strip())

# Check what DVC created
for f in sorted(os.listdir('.')):
    print(f'  {f}')


## 2. Create and Track a Data File

**Concept**: Generate a synthetic "dataset" (simulating CFD output). Use `dvc add` to start tracking it. Git will see a `.dvc` pointer file instead of the raw data.


In [ ]:
import numpy as np
import pandas as pd

# Simulate a medium-sized dataset (10 MB CSV)
np.random.seed(42)
n = 500_000
df = pd.DataFrame({
    'x': np.random.uniform(0, 1, n),
    'y': np.random.uniform(0, 1, n),
    'u': 1.0 + 0.5 * np.random.randn(n),
    'v': 0.1 * np.random.randn(n),
    'p': 101325 + 100 * np.random.randn(n),
})
df.to_csv('flow_data.csv', index=False)
size_mb = os.path.getsize('flow_data.csv') / 1e6
print(f'Created flow_data.csv: {size_mb:.1f} MB, {n:,} rows')


In [ ]:
# Add to DVC tracking
result = subprocess.run(['dvc', 'add', 'flow_data.csv'], capture_output=True, text=True)
print('DVC add:', result.stdout.strip() if result.returncode == 0 else result.stderr.strip())

# What files exist now?
print('\nFiles in directory:')
for f in sorted(os.listdir('.')):
    if not f.startswith('.'):
        size = os.path.getsize(f) if os.path.isfile(f) else 0
        print(f'  {f:30s} {size/1024:.0f} KB')

# Show the pointer file
if os.path.exists('flow_data.csv.dvc'):
    print('\nPointer file (flow_data.csv.dvc):')
    with open('flow_data.csv.dvc') as f:
        print(f.read())


## 3. What Git Actually Sees

**Key insight**: Git tracks the `.dvc` pointer (a few hundred bytes), NOT the 10 MB data file. The data file is in `.gitignore`.


In [ ]:
# Show what Git would track
result = subprocess.run(['git', 'status', '--short'], capture_output=True, text=True)
print('Git status:')
print(result.stdout)

# Show .gitignore (DVC adds entries)
if os.path.exists('.gitignore'):
    print('.gitignore:')
    with open('.gitignore') as f:
        for line in f:
            print(f'  {line.rstrip()}')


## 4. Version the Data

**Concept**: Modify the data (simulate a new simulation run). DVC detects the change by hash. Commit both the pointer and the data.


In [ ]:
# Simulate a new run (change the data)
df['u'] = 1.2 + 0.5 * np.random.randn(n)  # New mean velocity
df.to_csv('flow_data.csv', index=False)

# DVC detects the change
result = subprocess.run(['dvc', 'status'], capture_output=True, text=True)
print('DVC status (after modification):')
print(result.stdout)

# Re-add to update the pointer
result = subprocess.run(['dvc', 'add', 'flow_data.csv'], capture_output=True, text=True)
print('\nDVC add (updated):', result.stdout.strip() if result.returncode == 0 else result.stderr.strip())


## 5. Simulating a Remote: Push/Pull Workflow

**Concept**: In a real project, you set a remote (`dvc remote add -d myremote s3://bucket/path`). Here we demonstrate the local cache mechanism: the data lives in `.dvc/cache/`, and `dvc pull` restores it from cache.


In [ ]:
# The data is now in DVC's local cache
cache_dir = os.path.join(os.path.dirname(os.getcwd()), '.dvc', 'cache')
# Actually, cache is in the .dvc folder of the current dir
cache_dir = os.path.join(os.getcwd(), '.dvc', 'cache')

if os.path.exists(cache_dir):
    # Count cached files
    total_size = 0
    count = 0
    for root, dirs, files in os.walk(cache_dir):
        for f in files:
            total_size += os.path.getsize(os.path.join(root, f))
            count += 1
    print(f'DVC cache: {count} files, {total_size/1e6:.1f} MB')
    print('\nThis is where the actual data lives.')
    print('In a real project, dvc push sends this to a remote (S3, GDrive, SSH).')
    print('dvc pull retrieves it on another machine.')
else:
    print(f'Cache dir not found at {cache_dir}')
    print('Checking .dvc structure:')
    dvc_folder = os.path.join(os.getcwd(), '.dvc')
    if os.path.exists(dvc_folder):
        for f in os.listdir(dvc_folder):
            print(f'  .dvc/{f}')


## 6. Typical Engineering Workflow

```
# First time: set up the project
git init && dvc init
dvc remote add -d myremote s3://my-bucket/simulations
dvc add data/flow_data.csv
dvc add models/turbulence_model.pkl
git add .dvc/ .gitignore data/flow_data.csv.dvc models/turbulence_model.pkl.dvc
git commit -m "Add baseline flow data and trained model"
git push && dvc push

# New simulation run:
python run_simulation.py  # overwrites data/flow_data.csv
dvc add data/flow_data.csv  # updates pointer
git add data/flow_data.csv.dvc
git commit -m "Run 42: alpha=15, Re=1e6"
git push && dvc push

# Colleague reproduces:
git clone https://github.com/you/project
cd project && dvc install  # adds dvc hooks
dvc pull  # downloads all tracked data


## Practical Tips

| Situation | Recommendation |
|-----------|---------------|
| Dataset < 10 MB | Git LFS or even plain Git is fine |
| Dataset > 100 MB | DVC + S3/GDrive remote |
| Multiple contributors | DVC + shared remote, `dvc pull` after every `git pull` |
| HPC output (TB scale) | Partition data, DVC-track per partition |
| Models (pickled, ONNX) | DVC-track them, same as data |
| Reproducibility | Pin DVC version in `dvc.lock`, commit the `.dvc` pointers |

**Rule of thumb**: If a file is too big for Git (>50 MB) but you need version history for it, DVC is the answer.


## Exercises

1. **Set a real remote**: `dvc remote add -d local_remote /path/to/storage`. Push and pull to verify the round-trip.

2. **Version a model**: Train a simple scikit-learn model on your data, pickle it, `dvc add` it. Commit. Modify the model, re-add, commit. Check `git log` for the pointer changes.

3. **Reproduce**: On a fresh clone (or copy), run `dvc pull`. Verify the data matches the original (compare hashes).

4. **Large file**: Generate a 100 MB file. Compare Git performance with and without DVC tracking it.

5. **Integration**: Add DVC to your existing CFD project. Track your largest dataset. Commit. Push to a private GDrive remote.
